# Local-Evidence Whole-Text Classification

Train window classifiers, aggregate window MG scores into document predictions, and save a reusable artifact.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from cross_dataset_classification_utils import (
    DATASETS, EVALUATION_REGIMES, TRAINING_REGIMES, build_shared_parent_manifest,
    concat_or_empty, label_result_frame,
    source_coverage_row, summarize_bbc_false_positives,
)
from local_evidence_artifact_utils import save_local_evidence_artifact
from local_evidence_comparison_utils import expand_model_scores_to_tokens
from local_evidence_evaluation_utils import combine_local_evidence_dataset_results, evaluate_local_evidence_datasets
from local_evidence_models_utils import score_local_evidence_classifier
from local_evidence_training_utils import train_local_evidence_models
from local_evidence_utils import load_local_evidence_cohort, load_local_evidence_parent_metadata
from token_candidate_similarity_heatmaps_utils import load_token_labels
from whole_text_classification_utils import load_whole_text_cohort

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"
RUN_ID = "mixed-bbc-gemma-3-1b-v1"
SCORER_LLM = "gemma_3_1b_it"
TRAINING_REGIME = "Mixed+BBC"
SOURCE_SPLITS = {"BBC": ("HG",)}
CLASSIFIER_NAMES = ("logistic_regression",)
WINDOW_LENGTHS = (32, 64, 128)
STRIDE, TOP_K, SEED, MAX_PER_SPLIT = 16, 20, 42, 500
INSPECT_DATASET, INSPECT_SPLIT, INSPECT_TEXT_ID, INSPECT_WINDOW_LENGTH = "BBC", "HG", None, 64


In [ ]:
source_data, source_coverage = {}, []
for source_name, dataset_alias in DATASETS.items():
    source_splits = SOURCE_SPLITS.get(source_name, ("HG", "MG"))
    try:
        scalar = load_whole_text_cohort(PROCESSED_ROOT, dataset=dataset_alias, scorer_llm=SCORER_LLM)
        local_parents = load_local_evidence_parent_metadata(
            PROCESSED_ROOT, dataset_alias, SCORER_LLM, window_lengths=WINDOW_LENGTHS,
            stride=STRIDE, top_k=TOP_K, splits=source_splits,
        )
        manifest = build_shared_parent_manifest(scalar, local_parents, seed=SEED, max_per_split=MAX_PER_SPLIT)
        source_data[source_name] = {
            "alias": dataset_alias,
            "partitions": {name: manifest[name][["split", "text_id", "label"]] for name in manifest},
        }
        source_coverage.append({"source_dataset": source_name, "status": "available", "rows": len(scalar)})
    except (FileNotFoundError, KeyError, ValueError, OSError) as error:
        source_coverage.append(source_coverage_row(SCORER_LLM, source_name, error))

training_names = TRAINING_REGIMES[TRAINING_REGIME]
missing_training_sources = set(training_names).difference(source_data)
if missing_training_sources:
    raise RuntimeError(f"required training sources are unavailable: {sorted(missing_training_sources)}")

aliases = tuple(source_data[name]["alias"] for name in training_names)
trained = train_local_evidence_models(
    PROCESSED_ROOT, aliases, SCORER_LLM,
    {source_data[name]["alias"]: source_data[name]["partitions"]["train"] for name in training_names},
    {source_data[name]["alias"]: source_data[name]["partitions"]["validation"] for name in training_names},
    classifier_names=CLASSIFIER_NAMES, window_lengths=WINDOW_LENGTHS, stride=STRIDE, top_k=TOP_K, seed=SEED,
)
display(pd.DataFrame(source_coverage), trained.metrics, trained.coverage)

In [ ]:
metric_frames, prediction_frames, coverage_frames, scored_window_frames = [], [], [], []
for evaluation_regime, evaluation_names in EVALUATION_REGIMES.items():
    if not set(evaluation_names).issubset(source_data):
        continue
    for partition in ("train", "validation", "test"):
        separate = [
            evaluate_local_evidence_datasets(
                trained.model, PROCESSED_ROOT, (source_data[name]["alias"],),
                {source_data[name]["alias"]: source_data[name]["partitions"][partition]},
            )
            for name in evaluation_names
        ]
        evaluated = combine_local_evidence_dataset_results(separate, evaluation_regime=evaluation_regime) if evaluation_regime == "Mixed" else separate[0]
        for frame, frames, block in (
            (evaluated.metrics, metric_frames, "local_evidence"),
            (evaluated.predictions, prediction_frames, "local_evidence"),
            (evaluated.coverage, coverage_frames, "local_evidence"),
        ):
            frames.append(label_result_frame(
                frame, llm=SCORER_LLM, training_regime=TRAINING_REGIME,
                evaluation_regime=evaluation_regime, evaluation_dataset=("Mixed" if evaluation_regime == "Mixed" else ", ".join(evaluation_names)), block=block,
            ).assign(partition=partition))
        scored_window_frames.append(evaluated.scored_windows.assign(partition=partition, evaluation_regime=evaluation_regime))

evaluation_metrics = concat_or_empty(metric_frames)
evaluation_predictions = concat_or_empty(prediction_frames)
evaluation_coverage = concat_or_empty(coverage_frames)
scored_windows = concat_or_empty(scored_window_frames)
bbc_false_positive_summary = summarize_bbc_false_positives(evaluation_predictions, block="local_evidence")

ARTIFACT_METADATA = {
    "scorer_llm": SCORER_LLM, "training_regime": TRAINING_REGIME,
    "training_sources": list(training_names), "cohort_seed": SEED, "max_per_split": MAX_PER_SPLIT,
    "classifier_names": list(CLASSIFIER_NAMES), "window_lengths": list(WINDOW_LENGTHS),
    "stride": STRIDE, "top_k": TOP_K,
}
artifact_path = save_local_evidence_artifact(
    DATA_ROOT, RUN_ID, trained, metadata=ARTIFACT_METADATA,
    evaluation_results={"metrics": evaluation_metrics, "predictions": evaluation_predictions, "coverage": evaluation_coverage},
)
display(artifact_path, evaluation_metrics, bbc_false_positive_summary)

In [ ]:
if INSPECT_TEXT_ID is not None:
    dataset_alias = source_data[INSPECT_DATASET]["alias"]
    requested = pd.DataFrame([{"split": INSPECT_SPLIT, "text_id": INSPECT_TEXT_ID}])
    windows = load_local_evidence_cohort(
        PROCESSED_ROOT, dataset_alias, SCORER_LLM, requested, window_lengths=(INSPECT_WINDOW_LENGTH,), stride=STRIDE, top_k=TOP_K,
    )
    fitted = trained.model.classifiers[("logistic_regression", INSPECT_WINDOW_LENGTH)]
    scored = score_local_evidence_classifier(fitted, windows, trained.model.feature_columns)
    token_count = len(load_token_labels(PROCESSED_ROOT, dataset_alias, SCORER_LLM, INSPECT_SPLIT)[INSPECT_TEXT_ID])
    token_scores = expand_model_scores_to_tokens(scored, token_count)
    plt.figure(figsize=(14, 3))
    plt.plot(token_scores["token_position"], token_scores["mg_score"])
    plt.ylim(0, 1)
    plt.xlabel("Token position")
    plt.ylabel("MG score")
    plt.title(f"{INSPECT_DATASET} {INSPECT_SPLIT} {INSPECT_TEXT_ID}: token-level MG score")
    plt.show()
else:
    display(scored_windows.groupby(["evaluation_regime", "partition", "model", "window_length"]).size().rename("scored_windows").reset_index())